# ***FILM SETUP***

In [ ]:
import sys
print(sys.version)
print(sys.version_info)

In [ ]:
import shutil

backup_files = [
    "film_generic_correction.pth",
    "film_generic_correction_checkpoint.pth",
    "film_correction_rotation.pth",
    "film_correction_rotation_checkpoint.pth"
]

for fname in backup_files:
    src = os.path.join(BACKUP_DIR, fname)
    dst = os.path.join(CNN_OUT, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print("Restored:", fname)
    else:
        print("No backup found for:", fname)

print("\nCurrent contents of", CNN_OUT)
print(os.listdir(CNN_OUT))

In [ ]:
!pip install -q tensorflow==2.13.0 tensorflow-addons==0.21.0 gin-config mediapy

In [ ]:
import tensorflow as tf
import tensorflow_addons as tfa
import gin
import mediapy

print("TensorFlow:", tf.__version__)
print("TensorFlow Addons:", tfa.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
for gpu in tf.config.list_physical_devices("GPU"):
    print(gpu)

In [ ]:
import os
import shutil

SRC = "/kaggle/input/datasets/jaisai002/film-kaggle"
WORK = "/kaggle/working/film"

if os.path.exists(WORK):
    shutil.rmtree(WORK)

shutil.copytree(SRC, WORK)

print("FILM copied to:", WORK)
print("Top-level files/folders:")
print(os.listdir(WORK))

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/film")

from models.film_net import util
from eval import interpolator

print("FILM imports: OK")

In [ ]:
!pip install -q gdown

In [ ]:
import gdown
import os

MODEL_DIR = "/kaggle/working/pretrained_models"
os.makedirs(MODEL_DIR, exist_ok=True)

gdown.download_folder(
    "https://drive.google.com/drive/folders/1q8110-qp225asX3DQvZnfLfJPkCHmDpy",
    output=MODEL_DIR,
    quiet=False
)

print("\nDownload finished.")

In [ ]:
import os

MODEL_PATH = "/kaggle/working/pretrained_models/film_net/L1/saved_model"

print("Model exists:", os.path.exists(MODEL_PATH))
print("saved_model.pb:", os.path.exists(os.path.join(MODEL_PATH, "saved_model.pb")))
print("variables:", os.path.exists(os.path.join(MODEL_PATH, "variables")))
print("Model files:", os.listdir(MODEL_PATH))

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/film")

from eval import interpolator

MODEL_PATH = "/kaggle/working/pretrained_models/film_net/L1/saved_model"

print("Loading FILM...")
film = interpolator.Interpolator(MODEL_PATH, None)
print("FILM loaded successfully!")

In [ ]:
# Single-clip sanity check: frozen FILM baseline vs ground truth
import os
import numpy as np
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

VIMEO_ROOT = "/kaggle/input/datasets/chenshu123/vimeo-triplet/vimeo_triplet/sequences"
CLIP = "00001/0389"

base = os.path.join(VIMEO_ROOT, CLIP)

im0 = np.asarray(Image.open(os.path.join(base, "im1.png")).convert("RGB"), dtype=np.float32) / 255.0
gt  = np.asarray(Image.open(os.path.join(base, "im2.png")).convert("RGB"), dtype=np.float32) / 255.0
im1 = np.asarray(Image.open(os.path.join(base, "im3.png")).convert("RGB"), dtype=np.float32) / 255.0

im0_b = im0[None, ...]
im1_b = im1[None, ...]
dt = np.array([0.5], dtype=np.float32)

print("Input:", im0_b.shape)
print("GT:", gt.shape)

pred = film(im0_b, im1_b, dt)
pred = np.clip(np.asarray(pred)[0], 0.0, 1.0)

print("Prediction:", pred.shape)
print("Prediction range:", pred.min(), "to", pred.max())

psnr = peak_signal_noise_ratio(gt, pred, data_range=1.0)
ssim = structural_similarity(gt, pred, channel_axis=2, data_range=1.0)

print(f"PSNR: {psnr:.4f} dB")
print(f"SSIM: {ssim:.6f}")

# ***BATCH PROCESSING (SANITY CHECKS)***

In [ ]:
!pip install -q lpips

In [ ]:
import lpips
import torch

print("LPIPS:", lpips.__version__ if hasattr(lpips, "__version__") else "loaded")
print("CUDA available:", torch.cuda.is_available())

In [ ]:
import pandas as pd

SPLITS_PATH = "/kaggle/input/datasets/hibro00/vimeo-triplet-split/splits.csv"

splits = pd.read_csv(SPLITS_PATH)
test_ids = splits.loc[splits["split"] == "test", "clip_id"].tolist()

print("Test clips:", len(test_ids))
print(test_ids[:8])

In [ ]:
# 8-clip FILM batch inference smoke test (confirms batching works before the full run)
import os
import time
import numpy as np
from PIL import Image

clip_ids = test_ids[:8]

def load_triplet(clip_id):
    clip_dir = os.path.join(VIMEO_ROOT, clip_id)
    im0 = np.asarray(Image.open(f"{clip_dir}/im1.png").convert("RGB"), dtype=np.float32) / 255.0
    gt  = np.asarray(Image.open(f"{clip_dir}/im2.png").convert("RGB"), dtype=np.float32) / 255.0
    im1 = np.asarray(Image.open(f"{clip_dir}/im3.png").convert("RGB"), dtype=np.float32) / 255.0
    return im0, im1, gt

im0_list, im1_list, gt_list = [], [], []
for cid in clip_ids:
    im0, im1, gt = load_triplet(cid)
    im0_list.append(im0)
    im1_list.append(im1)
    gt_list.append(gt)

im0_b = np.stack(im0_list, axis=0)
im1_b = np.stack(im1_list, axis=0)
dt = np.full((len(clip_ids),), 0.5, dtype=np.float32)

print("Batch:", im0_b.shape)

t0 = time.perf_counter()
pred_batch = film(im0_b, im1_b, dt)
elapsed = time.perf_counter() - t0

pred_batch = np.clip(np.asarray(pred_batch), 0.0, 1.0)

print("Prediction:", pred_batch.shape)
print(f"Total FILM inference: {elapsed:.4f} sec")
print(f"Per-clip: {elapsed / len(clip_ids):.4f} sec")
print("GPU test successful.")

# ***CNN — MODEL & COMPONENTS***

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPUs:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

VIMEO_ROOT = "/kaggle/input/datasets/chenshu123/vimeo-triplet/vimeo_triplet/sequences"
CLASSIFIED_CSV = "/kaggle/input/datasets/hibro00/vfi-csvs/raft_descriptors_train_classified.csv"

CNN_OUT = "/kaggle/working/film_cnn"
os.makedirs(CNN_OUT, exist_ok=True)

NUM_GPUS = torch.cuda.device_count()
assert NUM_GPUS >= 2, (
    f"Only {NUM_GPUS} GPU(s) visible - the correction CNN needs two. "
    "Set the Kaggle accelerator to 'GPU T4 x2' and restart the session."
)

In [ ]:
classified = pd.read_csv(CLASSIFIED_CSV)

print("Shape:", classified.shape)
print(classified["predicted_category"].value_counts())

generic_df = classified.copy()
print("Generic clips:", len(generic_df))

In [ ]:
# Canonical dataset: returns (I0, I1, GT) for a clip_id + root dir
class CorrectionDataset(torch.utils.data.Dataset):
    def __init__(self, df, root):
        self.df = df.reset_index(drop=True)
        self.root = root

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        clip_id = self.df.iloc[idx]["clip_id"]
        base = os.path.join(self.root, clip_id)

        I0 = Image.open(os.path.join(base, "im1.png")).convert("RGB")
        GT = Image.open(os.path.join(base, "im2.png")).convert("RGB")
        I1 = Image.open(os.path.join(base, "im3.png")).convert("RGB")

        I0 = torch.from_numpy(np.array(I0, dtype=np.float32) / 255.0).permute(2, 0, 1)
        GT = torch.from_numpy(np.array(GT, dtype=np.float32) / 255.0).permute(2, 0, 1)
        I1 = torch.from_numpy(np.array(I1, dtype=np.float32) / 255.0).permute(2, 0, 1)

        return I0, I1, GT

In [ ]:
# Canonical model: forward(I0, I1, Ib) -> (corrected, residual, mask)
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1)
        )

    def forward(self, x):
        return x + self.block(x)


class CorrectionCNN(nn.Module):
    def __init__(self, base_channels=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(9, base_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            ResidualBlock(base_channels),
            ResidualBlock(base_channels),
            ResidualBlock(base_channels),
            nn.Conv2d(base_channels, base_channels, 3, padding=1),
            nn.ReLU(inplace=True)
        )
        self.residual_head = nn.Conv2d(base_channels, 3, 3, padding=1)
        self.mask_head = nn.Sequential(
            nn.Conv2d(base_channels, 1, 3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, I0, I1, Ib):
        x = torch.cat([I0, I1, Ib], dim=1)
        features = self.encoder(x)
        residual = self.residual_head(features)
        mask = self.mask_head(features)
        corrected = torch.clamp(Ib + mask * residual, 0, 1)
        return corrected, residual, mask

In [ ]:
def charbonnier_loss(pred, target, eps=1e-3):
    diff = pred - target
    return torch.mean(torch.sqrt(diff * diff + eps * eps))


def correction_loss(corrected, target, residual):
    return (
        F.l1_loss(corrected, target)
        + 0.1 * charbonnier_loss(corrected, target)
        + 0.001 * torch.mean(torch.abs(residual))
    )

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

correction_model = nn.DataParallel(
    CorrectionCNN(base_channels=32),
    device_ids=[0, 1]
).to(device)

optimizer = torch.optim.Adam(correction_model.parameters(), lr=1e-4)

print("Correction CNN ready")
print("Device:", device)
print("DataParallel GPUs:", correction_model.device_ids)

In [ ]:
def film_inference(I0, I1):
    x0 = I0.detach().cpu().numpy()
    x1 = I1.detach().cpu().numpy()

    x0 = np.transpose(x0, (0, 2, 3, 1)).astype(np.float32)
    x1 = np.transpose(x1, (0, 2, 3, 1)).astype(np.float32)

    dt = np.full((x0.shape[0],), 0.5, dtype=np.float32)

    pred = film(x0, x1, dt)
    pred = np.clip(np.asarray(pred), 0.0, 1.0)

    pred = torch.from_numpy(np.transpose(pred, (0, 3, 1, 2)))
    return pred

In [ ]:
# Smoke test: forward pass, then forward+backward (confirms both GPUs are used)
smoke_df = generic_df.sample(n=min(8, len(generic_df)), random_state=42)

smoke_loader = DataLoader(
    CorrectionDataset(smoke_df, VIMEO_ROOT),
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

I0, I1, GT = next(iter(smoke_loader))
I0 = I0.cuda(0, non_blocking=True)
I1 = I1.cuda(0, non_blocking=True)
GT = GT.cuda(0, non_blocking=True)

with torch.no_grad():
    Ib = film_inference(I0, I1).cuda(0)

corrected, residual, mask = correction_model(I0, I1, Ib)
loss = correction_loss(corrected, GT, residual)
print("Corrected:", corrected.shape, "| Mask:", mask.shape, "| Loss:", loss.item())

optimizer.zero_grad()
corrected, residual, mask = correction_model(I0, I1, Ib)
loss = correction_loss(corrected, GT, residual)
loss.backward()
optimizer.step()

print("Training step successful")
print("GPU 0:", torch.cuda.memory_allocated(0) / 1024**3, "GB")
print("GPU 1:", torch.cuda.memory_allocated(1) / 1024**3, "GB")

# ***CNN-GENERIC — TRAINING & EVALUATION***

In [ ]:
# Optional: restore a previously trained checkpoint from a backup dataset
# (skip this cell if you are training the generic model from scratch)
import shutil

SRC_DIR = "/kaggle/input/datasets/hibro00/vfi-pths-backup"
DST_DIR = CNN_OUT

backup_files = [
    "film_generic_correction.pth",
    "film_generic_correction_checkpoint.pth"
]

for f in backup_files:
    src = os.path.join(SRC_DIR, f)
    dst = os.path.join(DST_DIR, f)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print("Restored:", f)
    else:
        print("No backup found for:", f)

print("\nCurrent contents of", DST_DIR)
print(os.listdir(DST_DIR))

In [ ]:
SRC = "/kaggle/input/datasets/hibro00/vfi-csvs/raft_descriptors_train_classified.csv"
CATALOG_DIR = "/kaggle/working/catalog"
os.makedirs(CATALOG_DIR, exist_ok=True)

TRAIN_CSV = os.path.join(CATALOG_DIR, "raft_descriptors_train_classified.csv")
shutil.copy2(SRC, TRAIN_CSV)

print("Training CSV ready:", os.path.exists(TRAIN_CSV))

In [ ]:
# Optional diagnostic: inspect an existing checkpoint before (re)training
CHECKPOINT = os.path.join(CNN_OUT, "film_generic_correction_checkpoint.pth")
FINAL_MODEL = os.path.join(CNN_OUT, "film_generic_correction.pth")

if os.path.exists(CHECKPOINT):
    ckpt = torch.load(CHECKPOINT, map_location="cpu")
    if isinstance(ckpt, dict):
        print("Checkpoint keys:", list(ckpt.keys()))
        print("Epoch field:", ckpt.get("epoch", "NO EPOCH FIELD"))
        print("Loss field:", ckpt.get("loss", "NO LOSS FIELD"))
else:
    print("No checkpoint yet - training will start from epoch 1.")

In [ ]:
# ============================================
# FILM GENERIC CORRECTION TRAINING
# Reuses CorrectionCNN / CorrectionDataset / correction_model /
# optimizer / correction_loss / film_inference from the
# "CNN - MODEL & COMPONENTS" section above - no redefinitions here.
# ============================================
from tqdm import tqdm

EPOCHS = 5
BATCH_SIZE = 4
NUM_WORKERS = 2

train_df = pd.read_csv(TRAIN_CSV)
print("Training clips:", len(train_df))

train_loader = DataLoader(
    CorrectionDataset(train_df, VIMEO_ROOT),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False
)
print("Batches per epoch:", len(train_loader))

# --------------------------------------------------
# RESUME
# --------------------------------------------------
start_epoch = 0

if os.path.exists(CHECKPOINT):
    print("\nCheckpoint found:", CHECKPOINT)
    ckpt = torch.load(CHECKPOINT, map_location="cuda:0")

    if "model_state_dict" in ckpt:
        correction_model.load_state_dict(ckpt["model_state_dict"])
        if "optimizer_state_dict" in ckpt:
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])

        # "epoch" is the number of epochs already completed
        # (saved as `epoch + 1` at save time), so it is exactly
        # the correct 0-indexed start_epoch here.
        start_epoch = ckpt.get("epoch", 1)
        print(f"Checkpoint corresponds to completed Epoch {start_epoch}.")
        print(f"Resuming from Epoch {start_epoch + 1}.")
    else:
        correction_model.load_state_dict(ckpt)
        print("Raw model weights loaded (no epoch metadata) - assuming Epoch 1 complete.")
        start_epoch = 1
else:
    print("\nNo checkpoint found. Starting from Epoch 1.")

# --------------------------------------------------
# TRAIN
# --------------------------------------------------
print("\n===================================")
print("FILM GENERIC CORRECTION TRAINING")
print("===================================")
print("Total clips:", len(train_df))
print("Epochs:", EPOCHS)
print("Starting epoch:", start_epoch + 1)
print("Batch size:", BATCH_SIZE)
print("GPUs:", torch.cuda.device_count())

for epoch in range(start_epoch, EPOCHS):
    correction_model.train()
    running_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for I0, I1, GT in pbar:
        I0 = I0.cuda(0, non_blocking=True)
        I1 = I1.cuda(0, non_blocking=True)
        GT = GT.cuda(0, non_blocking=True)

        with torch.no_grad():
            Ib = film_inference(I0, I1).cuda(0, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        corrected, residual, mask = correction_model(I0, I1, Ib)
        loss = correction_loss(corrected, GT, residual)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        pbar.set_postfix(loss=f"{running_loss / (pbar.n + 1):.6f}")

    epoch_loss = running_loss / len(train_loader)
    print(f"\nEpoch {epoch+1}/{EPOCHS} COMPLETE")
    print(f"Loss: {epoch_loss:.6f}")

    torch.save(
        {
            "epoch": epoch + 1,
            "model_state_dict": correction_model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss": epoch_loss
        },
        CHECKPOINT
    )
    print("Checkpoint saved:", CHECKPOINT)

torch.save(correction_model.state_dict(), FINAL_MODEL)
print("\n===================================")
print("TRAINING FINISHED")
print("===================================")
print("Final model:", FINAL_MODEL)
print("Checkpoint:", CHECKPOINT)

In [ ]:
# Export with simple names into /kaggle/working (easy to find/download)
export_files = [FINAL_MODEL, CHECKPOINT]

for src in export_files:
    if os.path.exists(src):
        dst = "/kaggle/working/" + os.path.basename(src)
        shutil.copy2(src, dst)
        print("READY:", dst)
    else:
        print("MISSING:", src)

print("\n/kaggle/working contents:")
print(os.listdir("/kaggle/working"))

In [ ]:
# ============================================================
# FILM: BASELINE vs GENERIC CNN (test split)
# ============================================================
import lpips
from scipy import stats
from skimage.metrics import structural_similarity as sk_ssim

SPLITS_PATH     = "/kaggle/input/datasets/hibro00/vimeo-triplet-split/splits.csv"
TEST_CLASSIFIED = "/kaggle/input/datasets/hibro00/vfi-csvs/test_motion_classifications.csv"

GEN_W = os.path.join(CNN_OUT, "film_generic_correction.pth")
if not os.path.exists(GEN_W):
    GEN_W = "/kaggle/working/film_generic_correction.pth"

OUT_CSV   = "/kaggle/working/film_baseline_generic.csv"
EVAL_BS   = 8
MAX_CLIPS = None   # e.g. 64 for a quick trial run

assert os.path.exists(GEN_W), f"Missing weights: {GEN_W}"
print("Generic weights:", GEN_W)

# ------------------------------------------------------------
# LOAD CNN (handles full checkpoints and DataParallel prefixes)
# ------------------------------------------------------------
def load_cnn(path):
    sd = torch.load(path, map_location="cpu")
    if isinstance(sd, dict) and "model_state_dict" in sd:
        sd = sd["model_state_dict"]
    sd = {k[len("module."):] if k.startswith("module.") else k: v for k, v in sd.items()}
    net = CorrectionCNN(base_channels=32)
    net.load_state_dict(sd, strict=True)
    return net.to("cuda:0").eval()

gen_net = load_cnn(GEN_W)
lp = lpips.LPIPS(net="alex").to("cuda:0").eval()

# ------------------------------------------------------------
# TEST CLIPS
# ------------------------------------------------------------
sp = pd.read_csv(SPLITS_PATH)
sp["clip_id"] = sp["clip_id"].astype(str)
ids = list(dict.fromkeys(sp.loc[sp["split"].astype(str).str.lower() == "test", "clip_id"]))
if MAX_CLIPS:
    ids = ids[:MAX_CLIPS]
print("Test clips:", len(ids))

eval_loader = DataLoader(
    CorrectionDataset(pd.DataFrame({"clip_id": ids}), VIMEO_ROOT),
    batch_size=EVAL_BS, shuffle=False, num_workers=2, pin_memory=True,
)

# ------------------------------------------------------------
# BATCH METRICS
# ------------------------------------------------------------
def psnr_b(a, b):
    mse = ((a - b) ** 2).flatten(1).mean(1).clamp_min(1e-10)
    return (10 * torch.log10(1.0 / mse)).cpu().numpy()

def ssim_b(a, b):
    a = a.cpu().numpy().transpose(0, 2, 3, 1)
    b = b.cpu().numpy().transpose(0, 2, 3, 1)
    return np.array([sk_ssim(b[i], a[i], channel_axis=2, data_range=1.0) for i in range(len(a))])

def lpips_b(a, b):
    return lp(a * 2 - 1, b * 2 - 1).flatten().cpu().numpy()

# ------------------------------------------------------------
# EVALUATION LOOP
# ------------------------------------------------------------
rows, seen = [], 0
with torch.no_grad():
    for bi, (I0, I1, GT) in enumerate(tqdm(eval_loader, desc="FILM eval")):
        n = I0.shape[0]
        cids = ids[seen: seen + n]
        seen += n

        I0, I1, GT = I0.cuda(0), I1.cuda(0), GT.cuda(0)
        ib = film_inference(I0, I1).cuda(0).clamp(0, 1)      # frozen FILM baseline

        outs = {
            "base": ib,
            "gen":  gen_net(I0, I1, ib)[0].clamp(0, 1),
        }
        m = {k: (psnr_b(v, GT), ssim_b(v, GT), lpips_b(v, GT)) for k, v in outs.items()}

        for j, cid in enumerate(cids):
            r = {"clip_id": cid}
            for k in outs:
                r[f"{k}_psnr"]  = float(m[k][0][j])
                r[f"{k}_ssim"]  = float(m[k][1][j])
                r[f"{k}_lpips"] = float(m[k][2][j])
            rows.append(r)

        if (bi + 1) % 50 == 0:
            pd.DataFrame(rows).to_csv(OUT_CSV, index=False)

res = pd.DataFrame(rows)
assert len(res) == len(ids), f"Expected {len(ids)} rows, got {len(res)}"

# ------------------------------------------------------------
# CATEGORIES + DELTAS
# ------------------------------------------------------------
cat = pd.read_csv(TEST_CLASSIFIED)
cat["clip_id"] = cat["clip_id"].astype(str)
res = res.merge(cat[["clip_id", "predicted_category"]].drop_duplicates("clip_id"),
                on="clip_id", how="left").rename(columns={"predicted_category": "category"})
res["category"] = res["category"].fillna("unknown")

for met in ("psnr", "ssim", "lpips"):
    res[f"gen_d_{met}"] = res[f"gen_{met}"] - res[f"base_{met}"]

res.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV, "| clips:", len(res))

# ------------------------------------------------------------
# SUMMARY BY CATEGORY
# ------------------------------------------------------------
def summarize(g):
    return pd.Series({
        "n": len(g), "base_psnr": g["base_psnr"].mean(),
        "gen_d_psnr": g["gen_d_psnr"].mean(),
        "gen_d_ssim": g["gen_d_ssim"].mean(),
        "gen_d_lpips": g["gen_d_lpips"].mean(),
        "gen_win_%": 100 * (g["gen_d_psnr"] > 0).mean(),
    })

by_cat  = res.groupby("category").apply(summarize).reset_index()
overall = summarize(res).to_frame().T
overall.insert(0, "category", "ALL")
final = pd.concat([by_cat, overall], ignore_index=True)

print("\nd_* = corrected minus baseline. PSNR/SSIM: higher is better. LPIPS: lower is better.\n")
display(final.round(4))

# ------------------------------------------------------------
# SIGNIFICANCE (paired t-test vs 0)
# ------------------------------------------------------------
d = res["gen_d_psnr"]
print(f"\nGeneric PSNR delta: mean {d.mean():+.4f} dB | median {d.median():+.4f} | "
      f"p = {stats.ttest_1samp(d, 0).pvalue:.3g}")

In [ ]:
# ------------------------------------------------------------
# Original batch script for computing FILM's own baseline metrics
# (PSNR / SSIM / LPIPS / inference time) over the full test split.
# Output of this script is the film_metrics.csv used as an input
# dataset (hibro00/vfi-csvs/film_metrics.csv) elsewhere.
# ------------------------------------------------------------
# import os
# import time
# import json
# import logging
# import numpy as np
# import pandas as pd
# import torch
# from PIL import Image
# from skimage.metrics import peak_signal_noise_ratio, structural_similarity
# from tqdm.auto import tqdm
# import lpips as lpips_lib

# # ============================================================
# # PATHS
# # ============================================================
# VIMEO_ROOT = "/kaggle/input/datasets/chenshu123/vimeo-triplet/vimeo_triplet"
# SPLITS_PATH = "/kaggle/input/datasets/hibro00/vimeo-triplet-split/splits.csv"

# CATALOG_DIR = "/kaggle/working/catalog"
# LOG_DIR = "/kaggle/working/logs"

# os.makedirs(CATALOG_DIR, exist_ok=True)
# os.makedirs(LOG_DIR, exist_ok=True)

# OUTPUT_PATH = f"{CATALOG_DIR}/film_metrics.csv"
# LOG_PATH = f"{LOG_DIR}/film_batch_run.log"
# MANIFEST_PATH = f"{CATALOG_DIR}/film_manifest.json"

# # ============================================================
# # LOGGING
# # ============================================================
# logger = logging.getLogger("film_batch")
# logger.setLevel(logging.INFO)
# logger.handlers.clear()

# fh = logging.FileHandler(LOG_PATH)
# fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))

# ch = logging.StreamHandler()
# ch.setFormatter(logging.Formatter("%(message)s"))

# logger.addHandler(fh)
# logger.addHandler(ch)

# # ============================================================
# # SHARED TEST SPLIT
# # ============================================================
# splits = pd.read_csv(SPLITS_PATH)
# test_ids = splits.loc[splits["split"] == "test", "clip_id"].tolist()

# logger.info(f"Test clips: {len(test_ids)}")

# # ============================================================
# # LPIPS
# # ============================================================
# lpips_model = lpips_lib.LPIPS(net="alex").cuda().eval()
# logger.info("LPIPS loaded on CUDA.")

# # ============================================================
# # IMAGE LOADER
# # ============================================================
# def load_triplet(clip_id):
#     clip_dir = os.path.join(VIMEO_ROOT, clip_id)

#     im0 = np.asarray(
#         Image.open(os.path.join(clip_dir, "im1.png")).convert("RGB"),
#         dtype=np.float32
#     ) / 255.0

#     gt = np.asarray(
#         Image.open(os.path.join(clip_dir, "im2.png")).convert("RGB"),
#         dtype=np.float32
#     ) / 255.0

#     im1 = np.asarray(
#         Image.open(os.path.join(clip_dir, "im3.png")).convert("RGB"),
#         dtype=np.float32
#     ) / 255.0

#     return im0, im1, gt

# # ============================================================
# # RESUME
# # ============================================================
# existing = {}

# if os.path.exists(OUTPUT_PATH):
#     old_df = pd.read_csv(OUTPUT_PATH)

#     for _, row in old_df.iterrows():
#         if row["status"] == "success":
#             existing[row["clip_id"]] = row.to_dict()

#     logger.info(f"Existing successful clips: {len(existing)}")

# remaining_ids = [
#     cid for cid in test_ids
#     if cid not in existing
# ]

# logger.info(f"Remaining clips: {len(remaining_ids)}")

# # ============================================================
# # SETTINGS
# # ============================================================
# BATCH_SIZE = 4
# CHECKPOINT_EVERY_N_BATCHES = 5

# # ============================================================
# # RESULTS
# # ============================================================
# results = dict(existing)

# pbar = tqdm(
#     total=len(test_ids),
#     initial=len(existing),
#     unit="clip",
#     desc="FILM eval"
# )

# run_start = time.time()

# # ============================================================
# # BATCH LOOP
# # ============================================================
# for batch_num in range(
#     0,
#     len(remaining_ids),
#     BATCH_SIZE
# ):

#     clip_ids = remaining_ids[
#         batch_num:batch_num + BATCH_SIZE
#     ]

#     batch_index = batch_num // BATCH_SIZE + 1

#     try:

#         # ----------------------------------------------------
#         # LOAD DATA
#         # ----------------------------------------------------
#         im0_list = []
#         im1_list = []
#         gt_list = []

#         for cid in clip_ids:
#             im0, im1, gt = load_triplet(cid)

#             im0_list.append(im0)
#             im1_list.append(im1)
#             gt_list.append(gt)

#         im0_b = np.stack(im0_list, axis=0)
#         im1_b = np.stack(im1_list, axis=0)
#         gt_b = np.stack(gt_list, axis=0)

#         dt = np.full(
#             (len(clip_ids),),
#             0.5,
#             dtype=np.float32
#         )

#         # ----------------------------------------------------
#         # FILM INFERENCE — TIMED ONLY
#         # ----------------------------------------------------
#         t0 = time.perf_counter()

#         pred_b = film(
#             im0_b,
#             im1_b,
#             dt
#         )

#         elapsed = time.perf_counter() - t0

#         pred_b = np.clip(
#             np.asarray(pred_b),
#             0.0,
#             1.0
#         )

#         per_clip_time = elapsed / len(clip_ids)

#         # ----------------------------------------------------
#         # LPIPS
#         # ----------------------------------------------------
#         pred_t = (
#             torch.from_numpy(pred_b)
#             .permute(0, 3, 1, 2)
#             .cuda()
#             * 2.0 - 1.0
#         )

#         gt_t = (
#             torch.from_numpy(gt_b)
#             .permute(0, 3, 1, 2)
#             .cuda()
#             * 2.0 - 1.0
#         )

#         with torch.no_grad():
#             lpips_vals = (
#                 lpips_model(pred_t, gt_t)
#                 .flatten()
#                 .detach()
#                 .cpu()
#                 .numpy()
#             )

#         # ----------------------------------------------------
#         # METRICS
#         # ----------------------------------------------------
#         for i, cid in enumerate(clip_ids):

#             psnr = peak_signal_noise_ratio(
#                 gt_b[i],
#                 pred_b[i],
#                 data_range=1.0
#             )

#             ssim = structural_similarity(
#                 gt_b[i],
#                 pred_b[i],
#                 data_range=1.0,
#                 channel_axis=-1
#             )

#             results[cid] = {
#                 "clip_id": cid,
#                 "psnr": float(psnr),
#                 "ssim": float(ssim),
#                 "lpips": float(lpips_vals[i]),
#                 "inference_time_sec": float(per_clip_time),
#                 "status": "success"
#             }

#         pbar.update(len(clip_ids))

#         # ----------------------------------------------------
#         # CHECKPOINT
#         # ----------------------------------------------------
#         if batch_index % CHECKPOINT_EVERY_N_BATCHES == 0:

#             checkpoint_df = pd.DataFrame(
#                 [results[cid] for cid in test_ids
#                  if cid in results]
#             )

#             checkpoint_df.to_csv(
#                 OUTPUT_PATH,
#                 index=False
#             )

#             processed = len(results)
#             elapsed_total = time.time() - run_start

#             rate = (
#                 (processed - len(existing)) /
#                 elapsed_total
#                 if elapsed_total > 0 else 0
#             )

#             remaining = len(test_ids) - processed

#             eta = (
#                 remaining / rate
#                 if rate > 0 else float("nan")
#             )

#             logger.info(
#                 f"[{processed}/{len(test_ids)}] "
#                 f"rate={rate:.2f} clips/s "
#                 f"ETA={eta/60:.1f} min"
#             )

#     except Exception as e:

#         logger.error(
#             f"BATCH FAILED {batch_index}: {repr(e)}"
#         )

#         # Record failures, but they are NOT considered
#         # completed successful results and can be retried.
#         for cid in clip_ids:
#             results[cid] = {
#                 "clip_id": cid,
#                 "psnr": np.nan,
#                 "ssim": np.nan,
#                 "lpips": np.nan,
#                 "inference_time_sec": np.nan,
#                 "status": "failed"
#             }

#         pbar.update(len(clip_ids))

# pbar.close()

# # ============================================================
# # FINAL CSV
# # ============================================================
# final_df = pd.DataFrame(
#     [
#         results[cid]
#         for cid in test_ids
#         if cid in results
#     ]
# )

# final_df.to_csv(
#     OUTPUT_PATH,
#     index=False
# )

# # ============================================================
# # SUMMARY
# # ============================================================
# success_df = final_df[
#     final_df["status"] == "success"
# ]

# failed_df = final_df[
#     final_df["status"] == "failed"
# ]

# total_time = time.time() - run_start

# logger.info("========================================")
# logger.info("FILM BATCH COMPLETE")
# logger.info("========================================")

# logger.info(
#     f"Successful: {len(success_df)}"
# )

# logger.info(
#     f"Failed:     {len(failed_df)}"
# )

# if len(success_df) > 0:

#     logger.info(
#         f"Mean PSNR:  {success_df['psnr'].mean():.4f} dB"
#     )

#     logger.info(
#         f"Mean SSIM:  {success_df['ssim'].mean():.6f}"
#     )

#     logger.info(
#         f"Mean LPIPS: {success_df['lpips'].mean():.6f}"
#     )

#     logger.info(
#         f"Mean inference time: "
#         f"{success_df['inference_time_sec'].mean():.4f} sec/clip"
#     )

# logger.info(
#     f"Total benchmark time: {total_time/60:.2f} minutes"
# )

# logger.info(
#     f"Saved: {OUTPUT_PATH}"
# )

# # ============================================================
# # MANIFEST
# # ============================================================
# manifest = {
#     "model": "FILM",
#     "checkpoint": "film_net/L1/saved_model",
#     "dataset": "Vimeo-90K",
#     "split_source": SPLITS_PATH,
#     "test_count": len(test_ids),
#     "config": {
#         "batch_size": BATCH_SIZE,
#         "time": 0.5,
#         "lpips_network": "alex"
#     },
#     "counts": {
#         "total": int(len(final_df)),
#         "success": int(len(success_df)),
#         "failed": int(len(failed_df))
#     },
#     "files": {
#         "metrics": OUTPUT_PATH,
#         "log": LOG_PATH
#     }
# }

# with open(MANIFEST_PATH, "w") as f:
#     json.dump(manifest, f, indent=2)

# logger.info(
#     f"Manifest: {MANIFEST_PATH}"
# )

In [ ]:
# ------------------------------------------------------------
# Reusable training function (checkpointed, resumable, NaN/Inf-safe)
# used for the rotation-specialised correction model.
# ------------------------------------------------------------
def train_film_correction(
    df,
    model,
    epochs,
    checkpoint_path,
    final_path,
    batch_size=4
):
    import os
    import torch
    from torch.utils.data import DataLoader
    from tqdm import tqdm

    # --------------------------------------------------
    # DATA LOADER
    # --------------------------------------------------

    loader = DataLoader(
        CorrectionDataset(df, VIMEO_ROOT),
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        drop_last=True,
        persistent_workers=(NUM_WORKERS > 0)
    )

    print("\nStarting training...")
    print(f"Samples: {len(df)}")
    print(f"Batch size: {batch_size}")
    print(f"Batches per epoch: {len(loader)}")
    print(f"Epochs: {epochs}")
    print(f"Checkpoint: {checkpoint_path}")
    print()

    # --------------------------------------------------
    # RESUME
    # --------------------------------------------------

    start_epoch = 0
    best_loss = float("inf")

    if os.path.exists(checkpoint_path):

        print("Checkpoint found.")
        print("Loading checkpoint...")

        checkpoint = torch.load(
            checkpoint_path,
            map_location="cuda:0"
        )

        model.load_state_dict(
            checkpoint["model_state_dict"]
        )

        optimizer.load_state_dict(
            checkpoint["optimizer_state_dict"]
        )

        start_epoch = checkpoint["epoch"] + 1

        best_loss = checkpoint.get(
            "best_loss",
            float("inf")
        )

        print(
            f"Resuming from epoch "
            f"{start_epoch + 1}/{epochs}"
        )

        print(
            f"Previous best loss: "
            f"{best_loss:.6f}"
        )

    else:

        print("No checkpoint found.")
        print("Starting from epoch 1.")

    # --------------------------------------------------
    # TRAINING
    # --------------------------------------------------

    for epoch in range(start_epoch, epochs):

        model.train()

        epoch_loss = 0.0
        valid_batches = 0
        skipped = 0

        pbar = tqdm(
            loader,
            desc=f"Epoch {epoch + 1}/{epochs}",
            unit="batch",
            dynamic_ncols=True
        )

        for batch_idx, (I0, I1, GT) in enumerate(pbar):

            try:

                # --------------------------------------
                # MOVE DATA TO GPU 0
                # --------------------------------------

                I0 = I0.cuda(
                    0,
                    non_blocking=True
                )

                I1 = I1.cuda(
                    0,
                    non_blocking=True
                )

                GT = GT.cuda(
                    0,
                    non_blocking=True
                )

                # --------------------------------------
                # FROZEN FILM
                # --------------------------------------

                with torch.no_grad():

                    Ib = film_inference(
                        I0,
                        I1
                    )

                    Ib = Ib.cuda(
                        0,
                        non_blocking=True
                    )

                # --------------------------------------
                # CNN
                # --------------------------------------

                optimizer.zero_grad(
                    set_to_none=True
                )

                corrected, residual, mask = model(
                    I0,
                    I1,
                    Ib
                )

                # --------------------------------------
                # LOSS
                # --------------------------------------

                loss = correction_loss(
                    corrected,
                    GT,
                    residual
                )

                # --------------------------------------
                # FINITE CHECK
                # --------------------------------------

                if not torch.isfinite(loss):

                    skipped += 1

                    optimizer.zero_grad(
                        set_to_none=True
                    )

                    pbar.set_postfix(
                        loss="NaN/Inf",
                        skipped=skipped
                    )

                    continue

                # --------------------------------------
                # BACKPROP
                # --------------------------------------

                loss.backward()

                optimizer.step()

                # --------------------------------------
                # STATISTICS
                # --------------------------------------

                loss_value = loss.item()

                epoch_loss += loss_value
                valid_batches += 1

                running_loss = (
                    epoch_loss /
                    valid_batches
                )

                pbar.set_postfix(
                    loss=f"{loss_value:.5f}",
                    avg=f"{running_loss:.5f}",
                    skipped=skipped
                )

            # ------------------------------------------
            # CUDA OOM
            # ------------------------------------------

            except RuntimeError as e:

                if "out of memory" in str(e).lower():

                    optimizer.zero_grad(
                        set_to_none=True
                    )

                    torch.cuda.empty_cache()

                    raise RuntimeError(
                        f"\nCUDA OOM at batch "
                        f"{batch_idx + 1}.\n"
                        f"Current batch size: "
                        f"{batch_size}.\n"
                        f"Reduce batch size and restart."
                    )

                raise

        # --------------------------------------------------
        # EPOCH RESULTS
        # --------------------------------------------------

        if valid_batches > 0:

            avg_loss = (
                epoch_loss /
                valid_batches
            )

        else:

            avg_loss = float("inf")

        print()
        print("=" * 60)
        print(
            f"Epoch {epoch + 1}/{epochs} complete"
        )
        print(
            f"Average loss: {avg_loss:.6f}"
        )
        print(
            f"Valid batches: "
            f"{valid_batches}/{len(loader)}"
        )
        print(
            f"Skipped batches: {skipped}"
        )

        # --------------------------------------------------
        # CHECKPOINT
        # --------------------------------------------------

        checkpoint = {
            "epoch": epoch,
            "model_state_dict":
                model.state_dict(),
            "optimizer_state_dict":
                optimizer.state_dict(),
            "loss": avg_loss,
            "best_loss": min(
                best_loss,
                avg_loss
            )
        }

        torch.save(
            checkpoint,
            checkpoint_path
        )

        print(
            f"Checkpoint saved:\n"
            f"{checkpoint_path}"
        )

        # --------------------------------------------------
        # BEST MODEL
        # --------------------------------------------------

        if avg_loss < best_loss:

            best_loss = avg_loss

            torch.save(
                model.state_dict(),
                final_path
            )

            print(
                f"New best model saved:\n"
                f"{final_path}"
            )

        print("=" * 60)
        print()

        torch.cuda.empty_cache()

    # --------------------------------------------------
    # FINAL SAVE
    # --------------------------------------------------

    torch.save(
        model.state_dict(),
        final_path
    )

    print()
    print("=" * 60)
    print("TRAINING COMPLETE")
    print("=" * 60)
    print(
        f"Final model:\n{final_path}"
    )
    print(
        f"Checkpoint:\n{checkpoint_path}"
    )

In [ ]:
# ------------------------------------------------------------
# Rotation-specialised training config
# ------------------------------------------------------------
# ============================================================
# ROTATION-SPECIALISED FILM CORRECTION TRAINING
# ============================================================

import os
import math
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

BATCH_SIZE = 4
EPOCHS = 5
LR = 1e-4
NUM_WORKERS = 4

ROTATION_CKPT = os.path.join(
    CNN_OUT,
    "film_correction_rotation_checkpoint.pth"
)

ROTATION_FINAL = os.path.join(
    CNN_OUT,
    "film_correction_rotation.pth"
)

print("Training clips:", len(rotation_df))
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("GPUs:", correction_model.device_ids)

In [ ]:
# # ============================================================
# # FILM ROTATION-SPECIALISED CORRECTION TRAINING
# # ============================================================

# EPOCHS = 5
# BATCH_SIZE = 4

# ROTATION_CKPT = os.path.join(
#     CNN_OUT,
#     "film_correction_rotation_checkpoint.pth"
# )

# ROTATION_FINAL = os.path.join(
#     CNN_OUT,
#     "film_correction_rotation.pth"
# )

# print("Rotation clips:", len(rotation_df))

# rotation_cnn, rotation_history = train_film_correction(
#     df=rotation_df,
#     model=correction_model,
#     epochs=EPOCHS,
#     checkpoint_path=ROTATION_CKPT,
#     final_path=ROTATION_FINAL,
#     batch_size=BATCH_SIZE
# )

In [ ]:
# ------------------------------------------------------------
# Three-way evaluation: FILM baseline vs rotation-specialised CNN
# vs generic CNN. Superseded by the baseline-vs-generic evaluation
# above once the rotation model was dropped from the final pipeline.
# ------------------------------------------------------------
# # ============================================================
# # FILM: BASELINE vs ROTATION CNN vs GENERIC CNN (test split)
# # ============================================================
# import lpips
# import pandas as pd
# import numpy as np
# import torch
# from scipy import stats
# from tqdm import tqdm
# from torch.utils.data import DataLoader
# from skimage.metrics import structural_similarity as sk_ssim

# SPLITS_PATH       = "/kaggle/input/datasets/hibro00/vimeo-triplet-split/splits.csv"
# TEST_CLASSIFIED   = "/kaggle/input/datasets/hibro00/vfi-csvs/test_motion_classifications.csv"
# FILM_BASELINE_CSV = "/kaggle/input/datasets/hibro00/vfi-csvs/film_metrics.csv"

# ROT_W = os.path.join(CNN_OUT, "film_correction_rotation.pth")
# GEN_W = os.path.join(CNN_OUT, "film_generic_correction.pth")
# if not os.path.exists(GEN_W):
#     GEN_W = "/kaggle/working/film_generic_correction.pth"   # copy made in Cell 31

# OUT_CSV   = "/kaggle/working/film_baseline_rotation_generic.csv"
# EVAL_BS   = 8
# MAX_CLIPS = None        # e.g. 64 for a quick trial run

# for p in (ROT_W, GEN_W):
#     assert os.path.exists(p), f"Missing weights: {p}"
# print("Rotation weights:", ROT_W)
# print("Generic weights :", GEN_W)

# # ------------------------------------------------------------
# # LOAD CNNs (handles full checkpoints and DataParallel prefixes)
# # ------------------------------------------------------------
# def load_cnn(path):
#     sd = torch.load(path, map_location="cpu")
#     if isinstance(sd, dict) and "model_state_dict" in sd:
#         sd = sd["model_state_dict"]
#     sd = {k[len("module."):] if k.startswith("module.") else k: v for k, v in sd.items()}
#     net = CorrectionCNN(base_channels=32)
#     net.load_state_dict(sd, strict=True)
#     return net.to("cuda:0").eval()

# rot_net, gen_net = load_cnn(ROT_W), load_cnn(GEN_W)
# lp = lpips.LPIPS(net="alex").to("cuda:0").eval()

# # ------------------------------------------------------------
# # TEST CLIPS
# # ------------------------------------------------------------
# sp = pd.read_csv(SPLITS_PATH)
# sp["clip_id"] = sp["clip_id"].astype(str)
# ids = list(dict.fromkeys(sp.loc[sp["split"].astype(str).str.lower() == "test", "clip_id"]))
# if MAX_CLIPS:
#     ids = ids[:MAX_CLIPS]
# print("Test clips:", len(ids))

# loader = DataLoader(
#     CorrectionDataset(pd.DataFrame({"clip_id": ids}), VIMEO_ROOT),
#     batch_size=EVAL_BS, shuffle=False, num_workers=2, pin_memory=True,
# )

# # ------------------------------------------------------------
# # BATCH METRICS
# # ------------------------------------------------------------
# def psnr_b(a, b):
#     mse = ((a - b) ** 2).flatten(1).mean(1).clamp_min(1e-10)
#     return (10 * torch.log10(1.0 / mse)).cpu().numpy()

# def ssim_b(a, b):
#     a = a.cpu().numpy().transpose(0, 2, 3, 1)
#     b = b.cpu().numpy().transpose(0, 2, 3, 1)
#     return np.array([sk_ssim(b[i], a[i], channel_axis=2, data_range=1.0) for i in range(len(a))])

# def lpips_b(a, b):
#     return lp(a * 2 - 1, b * 2 - 1).flatten().cpu().numpy()

# # ------------------------------------------------------------
# # EVALUATION LOOP
# # ------------------------------------------------------------
# rows, seen = [], 0
# with torch.no_grad():
#     for bi, (I0, I1, GT) in enumerate(tqdm(loader, desc="FILM eval")):
#         n = I0.shape[0]
#         cids = ids[seen: seen + n]
#         seen += n

#         I0, I1, GT = I0.cuda(0), I1.cuda(0), GT.cuda(0)
#         ib = film_inference(I0, I1).cuda(0).clamp(0, 1)      # frozen FILM

#         outs = {
#             "base": ib,
#             "rot":  rot_net(I0, I1, ib)[0].clamp(0, 1),
#             "gen":  gen_net(I0, I1, ib)[0].clamp(0, 1),
#         }
#         m = {k: (psnr_b(v, GT), ssim_b(v, GT), lpips_b(v, GT)) for k, v in outs.items()}

#         for j, cid in enumerate(cids):
#             r = {"clip_id": cid}
#             for k in outs:
#                 r[f"{k}_psnr"]  = float(m[k][0][j])
#                 r[f"{k}_ssim"]  = float(m[k][1][j])
#                 r[f"{k}_lpips"] = float(m[k][2][j])
#             rows.append(r)

#         if (bi + 1) % 50 == 0:                                # crash-safe partial save
#             pd.DataFrame(rows).to_csv(OUT_CSV, index=False)

# res = pd.DataFrame(rows)
# assert len(res) == len(ids), f"Expected {len(ids)} rows, got {len(res)}"

# # ------------------------------------------------------------
# # CATEGORIES + DELTAS
# # ------------------------------------------------------------
# cat = pd.read_csv(TEST_CLASSIFIED)
# cat["clip_id"] = cat["clip_id"].astype(str)
# res = res.merge(cat[["clip_id", "predicted_category"]].drop_duplicates("clip_id"),
#                 on="clip_id", how="left").rename(columns={"predicted_category": "category"})
# res["category"] = res["category"].fillna("unknown")

# for k in ("rot", "gen"):
#     for met in ("psnr", "ssim", "lpips"):
#         res[f"{k}_d_{met}"] = res[f"{k}_{met}"] - res[f"base_{met}"]

# res.to_csv(OUT_CSV, index=False)
# print("Saved:", OUT_CSV, "| clips:", len(res))

# # ------------------------------------------------------------
# # SANITY CHECK vs film_metrics.csv (your original baseline)
# # ------------------------------------------------------------
# old = pd.read_csv(FILM_BASELINE_CSV)
# old["clip_id"] = old["clip_id"].astype(str)
# chk = res[["clip_id", "base_psnr", "base_ssim", "base_lpips"]].merge(
#     old[["clip_id", "psnr", "ssim", "lpips"]], on="clip_id")
# print(f"\nBaseline check vs film_metrics.csv ({len(chk)} clips):")
# print(f"  mean PSNR : in-pass {chk.base_psnr.mean():.4f} | csv {chk.psnr.mean():.4f} "
#       f"| diff {chk.base_psnr.mean() - chk.psnr.mean():+.4f} dB")
# print(f"  mean SSIM : in-pass {chk.base_ssim.mean():.5f} | csv {chk.ssim.mean():.5f}")
# print(f"  mean LPIPS: in-pass {chk.base_lpips.mean():.5f} | csv {chk.lpips.mean():.5f}")

# # ------------------------------------------------------------
# # SUMMARY BY CATEGORY
# # ------------------------------------------------------------
# def summarize(g):
#     return pd.Series({
#         "n": len(g), "base_psnr": g["base_psnr"].mean(),
#         "rot_d_psnr": g["rot_d_psnr"].mean(), "gen_d_psnr": g["gen_d_psnr"].mean(),
#         "rot_d_ssim": g["rot_d_ssim"].mean(), "gen_d_ssim": g["gen_d_ssim"].mean(),
#         "rot_d_lpips": g["rot_d_lpips"].mean(), "gen_d_lpips": g["gen_d_lpips"].mean(),
#         "rot_win_%": 100 * (g["rot_d_psnr"] > 0).mean(),
#         "gen_win_%": 100 * (g["gen_d_psnr"] > 0).mean(),
#     })

# by_cat  = res.groupby("category").apply(summarize).reset_index()
# overall = summarize(res).to_frame().T
# overall.insert(0, "category", "ALL")
# final = pd.concat([by_cat, overall], ignore_index=True)

# print("\nd_* = corrected minus baseline. PSNR/SSIM: higher is better. LPIPS: lower is better.\n")
# display(final.round(4))

# # ------------------------------------------------------------
# # SIGNIFICANCE (paired t-test vs 0 dB)
# # ------------------------------------------------------------
# print("\nOverall PSNR delta significance:")
# for k in ("rot", "gen"):
#     d = res[f"{k}_d_psnr"]
#     print(f"  {k}: mean {d.mean():+.4f} dB | median {d.median():+.4f} | "
#           f"p = {stats.ttest_1samp(d, 0).pvalue:.3g}")
# d = res["gen_d_psnr"] - res["rot_d_psnr"]
# print(f"  gen - rot: mean {d.mean():+.4f} dB | p = {stats.ttest_1samp(d, 0).pvalue:.3g}")